# Explore: NOMAD Perovskite Solar Cell Database

ตัวอย่างการดึงข้อมูลจากฐานข้อมูล perovskite solar cell ของ NOMAD แบบสด ๆ ผ่าน API
(คนละแหล่งกับไฟล์ Excel ที่เราคอมไพล์เองจากเปเปอร์ในโฟลเดอร์ `processed/`)

**แหล่งข้อมูล** (สองลิงก์นี้คือฐานข้อมูลเดียวกัน — เป็น schema plugin ของ NOMAD):
- https://nomad-lab.eu/prod/v1/gui/search/perovskite-solar-cells-database — หน้าค้นหาแบบ GUI
- https://pypi.org/project/perovskite-solar-cell-database/ — แพ็กเกจ Python ที่นิยาม schema ของข้อมูลชุดนี้

**ดึงข้อมูลยังไง:** แพ็กเกจ `perovskite-solar-cell-database` เอง **ไม่มี** ฟังก์ชันโหลดข้อมูลให้ตรง ๆ
มันแค่นิยาม schema (`PerovskiteSolarCell`) ที่ใช้กรองข้อมูลในฐานข้อมูล NOMAD
ตัวที่ใช้ดึงข้อมูลจริงคือ `ArchiveQuery` จากแพ็กเกจ `nomad-lab`
ซึ่งเป็น dependency หลักของ `perovskite-solar-cell-database` อยู่แล้ว (ติดมาด้วยตอน `pip install` ตัวเดียว)

โค้ดด้านล่างดัดแปลงจากตัวอย่างทางการ "Query the Perovskite Database"
(https://fairmat-nfdi.github.io/nomad-perovskite-solar-cells-database/notebooks/perla_notebooks/query-perovskite-database.html)
แต่ลดจำนวนที่ดึงให้เล็กลงมากสำหรับการ explore (ของจริงมีหลายหมื่น entry และเตือนว่าดาวน์โหลดทั้งหมดอาจใช้เวลาเป็นชั่วโมง)

## 1. ติดตั้ง

```bash
pip install perovskite-solar-cell-database nest_asyncio
```

`nest_asyncio` ต้องมีเพิ่ม เพราะ Jupyter มี event loop ของตัวเองอยู่แล้ว ต้องแปะ patch ให้ `await` ใช้ได้ในเซลล์
(แพ็กเกจหลักติดตั้ง `nomad-lab` และ dependency อื่น ๆ ให้อัตโนมัติ ใช้เวลาสักพักเพราะมีไลบรารีเคมี/วัสดุศาสตร์พ่วงมาเยอะ)

In [1]:
from time import time

import nest_asyncio
import pandas as pd
from nomad.client.archive import ArchiveQuery

nest_asyncio.apply()  # ให้ await ทำงานได้ในเซลล์ Jupyter

/Users/farlos/University/Perovskites-Research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. ตั้ง query

กรองเฉพาะ entry ที่เป็น perovskite solar cell (ด้วย schema qualified name) และวัดภายใต้แสงประมาณ 1 ซัน
(`illumination_intensity` ระหว่าง 600–1200 W/m²) ตามตัวอย่างทางการ

`results_max` / `page_size` ตั้งไว้ **เล็ก (20)** เพื่อ explore เร็ว ๆ — ถ้าจะดึงทั้งฐานข้อมูลจริง
ให้เพิ่มเลขนี้เป็นหลักหมื่น แล้วเตรียมใจรอ (API มี rate limit จะเจอ `429` เป็นระยะ ไลบรารี retry ให้เองแต่ช้า)

In [2]:
required = {
    "results": "*",
    "data": "*",
}

query = ArchiveQuery(
    query={
        "and": [
            {
                "results.properties.optoelectronic.solar_cell.illumination_intensity": {
                    "gte": 600.0,
                    "lte": 1200.0,
                }
            },
            {
                "section_defs.definition_qualified_name:all": [
                    "perovskite_solar_cell_database.schema.PerovskiteSolarCell"
                ]
            },
        ]
    },
    required=required,
    page_size=20,
    results_max=20,
)

## 3. นับจำนวนที่ตรงเงื่อนไข แล้วดาวน์โหลด

In [3]:
n_match = await query.async_fetch(20)
print(f"จำนวน entry ที่ตรงเงื่อนไข (indicative, จำกัดที่ขอไว้): {n_match}")

Fetching remote uploads...


20 entries are qualified and added to the download list.
จำนวน entry ที่ตรงเงื่อนไข (indicative, จำกัดที่ขอไว้): 20


In [4]:
t0 = time()
results = await query.async_download(20)
query._entries_dict.append(results)  # ต้องเติมเองตอนใช้ async mode (ตามตัวอย่างทางการ)
print(f"ดาวน์โหลดเสร็จใน {time() - t0:.1f} วินาที · ได้ {len(results)} entries")

Request returns 429, will retry in the next download call...


/Users/farlos/University/Perovskites-Research/.venv/lib/python3.12/site-packages/nomad/metainfo/metainfo.py:1419: SyntaxWarning: Atoms(atomic_numbers, species, labels, positions, lattice_vectors, periodic) is not derived from its definition nomad.datamodel.results.System.atoms:SubSection.
  warnings.warn(
/Users/farlos/University/Perovskites-Research/.venv/lib/python3.12/site-packages/nomad/metainfo/metainfo.py:1419: SyntaxWarning: ProvenanceTracker(label) is not derived from its definition nomad.datamodel.results.BandGap.provenance:SubSection.
  warnings.warn(


ดาวน์โหลดเสร็จใน 9.0 วินาที · ได้ 10 entries


## 4. แปลงเป็น DataFrame

`entries_to_dataframe()` แปะทุก field ของ schema มาเป็นคอลัมน์ — ได้มาหลักร้อยคอลัมน์ (schema นี้ละเอียดมาก
เก็บทั้งสารตั้งต้น, ตัวทำละลาย, อุณหภูมิอบ, ฯลฯ ของทุกเลเยอร์) ขั้นต่อไปค่อยเลือกเฉพาะคอลัมน์ที่สนใจ

In [5]:
df_raw = query.entries_to_dataframe()
df_raw.shape

(10, 680)

## 5. เลือกคอลัมน์ที่สนใจ (PV metric หลัก + องค์ประกอบ + metadata)

ชื่อคอลัมน์เต็มยาวมาก (เช่น `results.properties.optoelectronic.solar_cell.efficiency`)
ด้านล่างแมปเป็นชื่อสั้น ๆ อ่านง่ายขึ้น

**ข้อควรระวังเรื่องหน่วย:** field ใน `results.*` ถูก normalize เป็นหน่วย SI โดย NOMAD เอง
เช่น `short_circuit_current_density` เก็บเป็น **A/m²** ไม่ใช่ mA/cm² แบบที่เปเปอร์ perovskite รายงานกันทั่วไป
(ต้องหารด้วย 10 ถึงจะได้ mA/cm² — ไม่ใช่หน่วยเดียวกับที่เราใช้ในไฟล์ Excel ของโปรเจกต์นี้ ต้องแปลงก่อนเทียบ)

In [6]:
key_cols = {
    "data.ref.DOI_number": "doi",
    "results.properties.optoelectronic.solar_cell.efficiency": "pce_pct",
    "results.properties.optoelectronic.solar_cell.open_circuit_voltage": "voc_v",
    "results.properties.optoelectronic.solar_cell.short_circuit_current_density": "jsc_a_m2",
    "results.properties.optoelectronic.solar_cell.fill_factor": "ff",
    "results.properties.optoelectronic.solar_cell.illumination_intensity": "illumination_w_m2",
    "results.properties.optoelectronic.solar_cell.device_architecture": "device_architecture",
    "results.properties.optoelectronic.solar_cell.absorber": "absorber",
    "results.properties.optoelectronic.solar_cell.electron_transport_layer": "etl",
    "results.properties.optoelectronic.solar_cell.hole_transport_layer": "htl",
    "data.perovskite.composition_long_form": "composition_long",
    "data.perovskite.band_gap": "band_gap_ev",
}

df = df_raw[[c for c in key_cols if c in df_raw.columns]].rename(columns=key_cols)
df["jsc_ma_cm2"] = df["jsc_a_m2"] / 10  # A/m^2 -> mA/cm^2
df

,doi,pce_pct,voc_v,jsc_a_m2,ff,illumination_w_m2,device_architecture,absorber,etl,htl,composition_long,band_gap_ev,jsc_ma_cm2
0,10.1038/NPHOTON.2014.284,9.80,1.000,133.0,0.740,1000.0,pin,[MAPbI],"[PCBM-60, LiF]","[PEDOT:PSS, PDPP-DTT]",MAPbI3,1.6,13.30
1,10.1016/j.orgel.2019.105430,14.10,0.960,185.9,0.780,1000.0,pin,[MAPbI],"[PCBM-60, Bphen]",[PEDOT:PSS],MAPbI3,NaN,18.59
2,10.1021/ja307789s,5.50,0.632,161.0,0.570,1000.0,nip,[MAPbI],"[TiO2-c, TiO2-mp]",[none],MAPbI3,1.5,16.10
3,10.1016/j.electacta.2018.07.029,11.82,0.912,187.6,0.691,1000.0,pin,[MAPbI],"[PCBM-60, Bphen]",[PEDOT:PSS],MAPbI3,1.6,18.76
4,10.1002/asia.201501423,9.46,0.885,181.9,0.588,1000.0,nip,[MAPbI],[TiO2-c],[MeO-PheDOT],MAPbI3,1.6,18.19
5,10.1021/acsaem.9b00531,9.90,0.980,207.0,0.490,1000.0,nip,[MAPbI],[TiO2-c],[Spiro-MeOTAD],MAPbI3,1.6,20.70
6,10.1016/j.jpowsour.2018.04.086,18.67,1.040,235.0,0.762,1000.0,nip,[MAPbI],[TiO2-c],[Spiro-MeOTAD],MAPbI3,1.6,23.50
7,10.1016/j.mssp.2019.104798,6.17,0.776,154.3,0.510,1000.0,nip,[MAPbI],"[TiO2-c, TiO2-mp]",[none],MAPbI3,1.6,15.43
8,https://www.doi.org/10.1007/s10854-024-13107-0,1.23,0.532,45.6,0.506,1000.0,nip,[MAPbClI],[TiO2-c],NaN,MAPbClxI3-x,1.55,4.56
9,10.1016/j.nanoen.2015.06.010,12.70,NaN,NaN,NaN,1000.0,pin,[MAPbI],[PCBM-60],[PEDOT:PSS],MAPbI3,1.6,NaN


## 6. ดูสถิติเบื้องต้นของ metric หลัก

In [7]:
df[["pce_pct", "voc_v", "jsc_ma_cm2", "ff", "band_gap_ev"]].describe()

,pce_pct,voc_v,jsc_ma_cm2,ff
count,10.000000,9.000000,9.000000,9.000000
mean,9.935000,0.857444,16.570000,0.626333
std,4.884357,0.175462,5.397178,0.117454
min,1.230000,0.532000,4.560000,0.490000
25%,6.992500,0.776000,15.430000,0.510000
50%,9.850000,0.912000,18.190000,0.588000
75%,12.480000,0.980000,18.760000,0.740000
max,18.670000,1.040000,23.500000,0.780000


ตัวเลขนี้ไม่ได้ใช้สรุปอะไร — เป็นแค่ sample 10 entry เพื่อเช็กว่าดึงข้อมูลได้จริงและหน่วยถูกต้อง
(ถ้าจะวิเคราะห์เทรนด์จริงต้องดึงมากกว่านี้มาก และยังต้องเช็ก physics-sanity แบบที่ทำกับข้อมูลของเราเอง
เช่น PCE ≈ Voc × Jsc × FF, FF ไม่เกิน 0.89, ไม่เกิน Shockley–Queisser limit)

## ขั้นต่อไป (ถ้าจะใช้จริง)

- **ดึงเพิ่ม:** ขยับ `page_size` / `results_max` ขึ้น (หลักพัน–หมื่น) แล้วรันซ้ำ — รอได้นาน เจอ `429` เป็นปกติ
- **เซฟไว้ใช้ต่อ** โดยไม่ต้องยิง API ใหม่ทุกครั้ง: `df.to_parquet("nomad_sample.parquet", index=False)`
- **ตัวอย่างทางการอื่น ๆ ที่เกี่ยวข้อง** (https://fairmat-nfdi.github.io/nomad-perovskite-solar-cells-database/tutorial/):
  Query the Ion Database, Query Perovskite Compositions, Bandgap/Performance Evolution Analysis
- ข้อมูลชุดนี้เป็นฐานข้อมูลสาธารณะจากเปเปอร์ทั่วโลก (โครงสร้างคล้ายของเราแต่คนละชุด) — ถ้าจะเทียบ/รวมกับข้อมูลใน `processed/`
  ของโปรเจกต์นี้ ต้องระวังเรื่อง schema ไม่ตรงกัน (ชื่อคอลัมน์, หน่วย, การตั้งชื่อเลเยอร์) เหมือนที่เจอตอน prepare ข้อมูลของเราเอง